# E1 — Free-motion transparency

Sinusoids along the heaviest and lightest directions of Λ₀ (`e1_free_motion.py`). Expected: M1's e_F
grows with the delay T; M3's grows with the frequency (it renders no inertia); M5 stays low; M4, retuned
per direction, should match M5 in these two directions. On the robot, friction sets M5's floor.

In [ ]:
# Runs to analyse: run directories, session files (exp/<exp>/sessions/*.txt, written by the script) or a
# folder of runs. Default: every e1 run on the robot.
RUNS = ["~/data/franka/rim-delays/robot/e1"]
RECOMPUTE = False  # True: recompute the metrics from the data instead of reading analysis/metrics.json

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()  # experiments/rim-delays/exp
sys.path[:0] = [str(HERE), str(HERE.parent)]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import exp_metrics
import plots as P
from common import collect, load_table

%matplotlib inline
pd.set_option("display.precision", 3)

runs = collect(RUNS)
df = load_table(runs, exp_metrics.e1 if RECOMPUTE else None)
print(f"{len(df)} runs")

In [ ]:
df["direction"] = df.label.str.split("_").str[0]
df[["run", "method", "rtt", "direction", "frequency_hz", "accel_peak", "eF_n_rms", "eF_rms", "f_n_rms", "Fosc_n_rms",
    "lag_ms", "E_end_J"]].sort_values(["direction", "frequency_hz", "method", "rtt"])

## RMS e_F against the delay
One panel per direction and frequency; one line per method.

In [ ]:
dirs, freqs = sorted(df.direction.unique()), sorted(df.frequency_hz.unique())
fig, axes = plt.subplots(len(dirs), len(freqs), figsize=(4 * len(freqs), 3.2 * len(dirs)), squeeze=False, sharey=True)
for i, d in enumerate(dirs):
    for j, f in enumerate(freqs):
        ax = axes[i, j]
        P.by_delay(ax, df[(df.direction == d) & (df.frequency_hz == f)], "eF_n_rms", label_methods=(i == 0 and j == 0))
        ax.set_title(f"{d} direction, {f:g} Hz", loc="left", fontsize=10)
    axes[i, 0].set_ylabel("RMS e_F along the motion [N]")
fig.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.04))
fig.tight_layout()

## RMS e_F against the frequency
At each delay; the felt force `f` is shown dashed for scale.

In [ ]:
delays = [d for d in P.DELAYS if d in set(df.rtt.map(P.delay_key))]
fig, axes = plt.subplots(len(dirs), len(delays), figsize=(4 * len(delays), 3.2 * len(dirs)), squeeze=False, sharey=True)
for i, d in enumerate(dirs):
    for j, T in enumerate(delays):
        ax = axes[i, j]
        sub = df[(df.direction == d) & (df.rtt.map(P.delay_key) == T)]
        for m in P.methods_in(sub):
            s = sub[sub.method == m].groupby("frequency_hz")[["eF_n_rms", "f_n_rms"]].mean()
            ax.plot(s.index, s.eF_n_rms, "o-", color=P.color(m), label=P.LABELS.get(m, m) if (i, j) == (0, 0) else None)
        ax.set_title(f"{d} direction, T = {T}", loc="left", fontsize=10)
        ax.set_xlabel("frequency [Hz]")
    axes[i, 0].set_ylabel("RMS e_F along the motion [N]")
fig.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.04))
fig.tight_layout()